<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs the core data and visualization libraries needed for the notebook.

In [ ]:
!pip install pandas pandas_datareader matplotlib pyfolio

Zipline requires special installation through conda or a maintained fork like zipline-reloaded, because it depends on compiled C extensions and specific NumPy/pandas version constraints. Run `conda install -c conda-forge zipline-reloaded` separately before using this notebook.

## Imports and setup

We use pandas for timestamps and data manipulation, pandas_datareader to pull S&P 500 benchmark data from FRED, and matplotlib for any charts. Zipline provides the backtesting engine itself, including its order execution API, commission models, and slippage models. PyFolio takes Zipline's output and generates professional performance reports.

In [ ]:
import pandas as pd
import pandas_datareader.data as web
import matplotlib.pyplot as plt
from zipline import run_algorithm
from zipline.api import order_target, record, symbol
from zipline.finance import commission, slippage
import pyfolio as pf
import warnings
warnings.filterwarnings('ignore')

## Define the strategy and cost models

The initialize function runs once at the start of the backtest and sets up everything the strategy needs, including the asset to trade and the cost models that make our results realistic.

In [ ]:
def initialize(context):
    context.i = 0
    context.asset = symbol("AAPL")

    context.set_commission(commission.PerShare(cost=0.01))
    context.set_slippage(slippage.FixedSlippage(spread=0.01))

Setting commission and slippage explicitly is the single most important step that separates a professional backtest from a toy one. The PerShare commission charges $0.01 per share on every trade, and FixedSlippage adds a $0.01 spread to simulate the price impact of our orders. Without these two lines, Zipline would assume zero trading costs, which is exactly the mistake the post warns about.

The handle_data function runs once per trading day and contains our actual strategy logic. It waits 50 days to accumulate enough price history, then computes a 14-day and 50-day moving average to decide whether to buy or sell.

In [ ]:
def handle_data(context, data):
    context.i += 1
    if context.i < 50:
        return

    short_mavg = data.history(
        context.asset,
        "price",
        bar_count=14,
        frequency="1d"
    ).mean()

    long_mavg = data.history(
        context.asset,
        "price",
        bar_count=50,
        frequency="1d"
    ).mean()

    if short_mavg > long_mavg:
        order_target(context.asset, 100)
    elif short_mavg < long_mavg:
        order_target(context.asset, 0)

This is a classic moving average crossover strategy. When the short-term average crosses above the long-term average, we buy 100 shares; when it crosses below, we sell everything. The function order_target handles the math of figuring out how many shares to buy or sell to reach our target position, which mirrors how real brokers process orders rather than assuming instant, perfect fills.

## Run the backtest with benchmark data

We define the backtest period, pull S&P 500 data from FRED as our benchmark, and run the full simulation with $100,000 in starting capital using Zipline's Quandl data bundle.

In [ ]:
start = pd.Timestamp('2000')
end = pd.Timestamp('2018')

In [ ]:
sp500 = web.DataReader('SP500', 'fred', start, end).SP500
benchmark_returns = sp500.pct_change()

In [ ]:
perf = run_algorithm(
    start=start,
    end=end,
    initialize=initialize,
    handle_data=handle_data,
    capital_base=100000,
    benchmark_returns=benchmark_returns,
    bundle="quandl",
    data_frequency="daily",
)

The run_algorithm call is where Zipline replays 18 years of daily market data and simulates every trade our strategy would have made, including the commission and slippage costs we defined earlier. Comparing against S&P 500 benchmark returns lets us see whether our strategy actually adds value or just rides the broader market. The "quandl" bundle refers to a pre-ingested dataset of US equity prices that Zipline uses as its historical data source.

## Generate the performance tear sheet

We extract returns, positions, and transactions from Zipline's output and pass them to PyFolio to generate a comprehensive performance report.

In [ ]:
returns, positions, transactions = \
    pf.utils.extract_rets_pos_txn_from_zipline(perf)

In [ ]:
pf.create_full_tear_sheet(
    returns,
    positions=positions,
    transactions=transactions,
    live_start_date="2016-01-01",
    round_trips=True,
)

The tear sheet gives us cumulative returns, drawdowns, rolling volatility, and round-trip trade analysis all in one view. Setting live_start_date to 2016 splits the report into an in-sample period (2000-2015) and an out-of-sample period (2016-2018), which helps us see whether the strategy holds up on data it wasn't originally tested against. This kind of out-of-sample validation is how professionals catch overfitting before it costs real money.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.